# 05 — Vector Store

**Module notebook — definitions only.**

Builds/loads the Chroma vector store used for RAG over the transcript.

Depends on: `os` (loaded in `00_llm_config.ipynb`).

> ## ⚠️ Legacy / not used by the live pipeline
>
> **the LangChain/Chroma retriever this notebook builds is no longer used by `09_agents.ipynb` or the LangGraph
> orchestrator (`11_orchestrator.ipynb`).** It was replaced by
> `08_llamaindex_retriever.ipynb`, which handles RAG retrieval for the
> live `rag_agent`. `content_agent`/`rag_agent` in `09_agents.ipynb` call
> `build_llama_index_bundle()` / `ask_llama_question()` from `08`, never
> anything from this notebook.
>
> This notebook is kept only because `main.ipynb`'s standalone
> `run_pipeline()` helper (a manual-testing convenience, separate from
> the LangGraph app) still uses it. If that helper is ever removed too,
> this notebook -- and the `langchain-chroma` / `chromadb` /
> `langchain-huggingface` dependencies it pulls in -- can be deleted
> entirely.


In [ ]:
import shutil
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings  # not the deprecated langchain_community path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

CHROMA_DIR = "vector_db"
COLLECTION_NAME = "meeting_transcript"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
VECTOR_CHUNK_SIZE = 500
VECTOR_CHUNK_OVERLAP = 50


In [ ]:
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
    )


In [ ]:
def build_vector_store(transcript: str, reset: bool = True) -> Chroma:
    """
    Build a fresh Chroma store for this transcript.

    reset=True (default) wipes any existing collection at CHROMA_DIR first, so
    repeated runs on different transcripts don't silently mix old and new
    context in the same collection.
    """
    print("Building vector store")

    if reset and os.path.exists(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=VECTOR_CHUNK_SIZE,
        chunk_overlap=VECTOR_CHUNK_OVERLAP,
    )
    chunks = splitter.split_text(transcript)

    docs = [
        Document(page_content=chunk, metadata={"chunk_index": i})
        for i, chunk in enumerate(chunks)
    ]

    embeddings = get_embeddings()
    vector_store = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR,
    )

    return vector_store


In [ ]:
def load_vector_store() -> Chroma:
    embeddings = get_embeddings()
    return Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )


def get_retriever(vector_store: Chroma, k: int = 4):
    return vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )
